# Autoencoder performance — baseline (2026-05-07)

Waveforms → encoder → predicted params → frozen decoder → reconstructed waveforms.

Per-channel MAE, Median AE, MAPE (mean & median); waveform inspection for fixed and random test examples.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from dataset import CVDataset
from decoder import CVSurrogate
from encoder import CVEncoder

In [ ]:
ENCODER_CKPT = ROOT / 'checkpoints/autoenc-baseline/checkpoint_017.pt'
STATS_PATH   = ROOT / 'norm_stats.json'
DATA_DIR     = Path('/media/8TBNVME/data/neh10/hdf5/cv8/simset_10M_cv8Eed_20260314/test')
MANIFEST     = DATA_DIR.parent / 'manifest_test.json'
N_SIMS       = 256
SIM_INDICES  = [0, 1, 2]
FIGURES_DIR  = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

WAVE_NAMES_CONT = [
    'Pap','Pas','Pla','Plv','Pra','Prv','Pvp','Pvs',
    'Qap','Qas','Qla','Qlv','Qra','Qrv','Qvp','Qvs',
    'Vap','Vas','Vla','Vlv','Vra','Vrv','Vvp','Vvs',
]
VALVE_NAMES = ['AV', 'MV', 'PV', 'TV']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Load models

In [ ]:
enc_ckpt = torch.load(ENCODER_CKPT, map_location=device)

encoder = CVEncoder().to(device)
encoder.load_state_dict(enc_ckpt['model_state'])
encoder.eval()

decoder_path = ROOT / enc_ckpt['decoder_ckpt']
dec_ckpt = torch.load(decoder_path, map_location=device)
decoder = CVSurrogate().to(device)
decoder.load_state_dict(dec_ckpt['model_state'])
decoder.eval()

print(f"Encoder  epoch {enc_ckpt['epoch']}  val_loss={enc_ckpt['val_loss']:.6f}")
print(f"Decoder  epoch {dec_ckpt['epoch']}  val_loss={dec_ckpt['val_loss']:.6f}  (frozen)")

with open(STATS_PATH) as f:
    stats = json.load(f)
with open(MANIFEST) as f:
    manifest = json.load(f)

## Run inference on test set

In [ ]:
ds     = CVDataset(DATA_DIR, manifest['index'][:N_SIMS], stats)
loader = DataLoader(ds, batch_size=256, num_workers=0)

all_pred, all_gt = [], []
with torch.no_grad():
    for _, waves_cont_b, waves_valve_b in loader:
        pred_params = encoder(waves_cont_b.to(device), waves_valve_b.to(device))
        pred_cont_b, _ = decoder(pred_params)
        all_pred.append(pred_cont_b.cpu())
        all_gt.append(waves_cont_b)
ds.close()

all_pred = torch.cat(all_pred).numpy()   # (N, 24, 201)  — z-scored
all_gt   = torch.cat(all_gt).numpy()

wave_means = np.array([stats['waves'][n]['mean'] for n in WAVE_NAMES_CONT])
wave_stds  = np.array([stats['waves'][n]['std']  for n in WAVE_NAMES_CONT])
pred_phys  = all_pred * wave_stds[None, :, None] + wave_means[None, :, None]
gt_phys    = all_gt   * wave_stds[None, :, None] + wave_means[None, :, None]
print(f'Evaluated {N_SIMS} test simulations')

## Compute metrics

In [ ]:
abs_err = np.abs(pred_phys - gt_phys)          # (N, 24, 201)

# MAE — mean and median over sims x time
sim_ch_mae  = abs_err.mean(axis=2)             # (N, 24)
ch_mae      = sim_ch_mae.mean(axis=0)          # (24,)
ch_med_ae   = np.median(abs_err, axis=(0, 2))  # (24,)

# MAPE — mean and median over sims x time
pct_err      = abs_err / (np.abs(gt_phys) + 1e-6) * 100
sim_ch_mape  = pct_err.mean(axis=2)            # (N, 24)
ch_mape      = sim_ch_mape.mean(axis=0)        # (24,)
ch_med_mape  = np.median(pct_err, axis=(0, 2)) # (24,)

# Range-normalised error — mean and median (safe for zero-crossing channels)
sig_range    = np.maximum(gt_phys.max(axis=2) - gt_phys.min(axis=2), 1e-6)  # (N, 24)
rng_pct_err  = abs_err / sig_range[:, :, None] * 100
sim_ch_rne   = rng_pct_err.mean(axis=2)        # (N, 24)
ch_rne       = sim_ch_rne.mean(axis=0)         # (24,)
ch_med_rne   = np.median(rng_pct_err, axis=(0, 2))  # (24,)

print(f"{'Channel':<8} {'Mean MAE':>10} {'Med MAE':>10} {'Mean MAPE':>11} {'Med MAPE':>10} {'Mean RNE':>10} {'Med RNE':>9}")
print('-' * 75)
for i, name in enumerate(WAVE_NAMES_CONT):
    print(f'{name:<8} {ch_mae[i]:>10.4f} {ch_med_ae[i]:>10.4f} {ch_mape[i]:>10.2f}% {ch_med_mape[i]:>9.2f}% {ch_rne[i]:>9.2f}% {ch_med_rne[i]:>8.2f}%')

## Error distributions

In [ ]:
x = np.arange(len(WAVE_NAMES_CONT))
w = 0.4
fig, axes = plt.subplots(2, 2, figsize=(20, 10))

# MAE
axes[0, 0].bar(x - w/2, ch_mae,    width=w, color='steelblue', alpha=0.85, label='Mean MAE')
axes[0, 0].bar(x + w/2, ch_med_ae, width=w, color='darkcyan',  alpha=0.85, label='Median MAE')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(WAVE_NAMES_CONT, rotation=45, ha='right')
axes[0, 0].set_title('MAE — mean vs median per channel')
axes[0, 0].set_ylabel('AE (physical units)')
axes[0, 0].legend()

# MAPE
axes[0, 1].bar(x - w/2, ch_mape,     width=w, color='tomato',     alpha=0.85, label='Mean MAPE')
axes[0, 1].bar(x + w/2, ch_med_mape, width=w, color='darkorange',  alpha=0.85, label='Median MAPE')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(WAVE_NAMES_CONT, rotation=45, ha='right')
axes[0, 1].set_title('MAPE — mean vs median  (inflated for zero-crossing Q/V)')
axes[0, 1].set_ylabel('MAPE (%)')
axes[0, 1].legend()

# Range-normalised error
axes[1, 0].bar(x - w/2, ch_rne,     width=w, color='mediumseagreen', alpha=0.85, label='Mean RNE')
axes[1, 0].bar(x + w/2, ch_med_rne, width=w, color='darkgreen',      alpha=0.85, label='Median RNE')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(WAVE_NAMES_CONT, rotation=45, ha='right')
axes[1, 0].set_title('Range-normalised error — mean vs median  (safe for zero-crossing)')
axes[1, 0].set_ylabel('RNE (%)')
axes[1, 0].legend()

axes[1, 1].axis('off')

fig.suptitle(f'Autoenc-baseline — error distributions  (epoch {enc_ckpt["epoch"]}, n={N_SIMS})', fontsize=13)
plt.tight_layout()
plt.show()

## Waveforms — fixed test examples (sims 0, 1, 2)

In [ ]:
def run_samples(indices):
    ds_ = CVDataset(DATA_DIR, [manifest['index'][i] for i in indices], stats)
    samples_ = [ds_[j] for j in range(len(indices))]
    ds_.close()
    pred_c, pred_v, gt_c, gt_v = [], [], [], []
    for _, wc, wv in samples_:
        with torch.no_grad():
            pp = encoder(wc.unsqueeze(0).to(device), wv.unsqueeze(0).to(device))
            pc, pv = decoder(pp)
        pred_c.append(pc.squeeze(0).cpu().numpy())
        pred_v.append(pv.squeeze(0).cpu().numpy())
        gt_c.append(wc.numpy())
        gt_v.append(wv.numpy())
    return pred_c, pred_v, gt_c, gt_v

t = np.arange(201)
pred_conts, pred_valves, gt_conts, gt_valves = run_samples(SIM_INDICES)

In [ ]:
def plot_continuous(indices, pred_c, gt_c, title_suffix=''):
    ncols = len(indices)
    nrows = len(WAVE_NAMES_CONT)
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 2.5 * nrows), sharex=True)
    for col, idx in enumerate(indices):
        for row, name in enumerate(WAVE_NAMES_CONT):
            ax = axes[row, col]
            ax.plot(t, gt_c[col][row], color='steelblue', label='GT')
            ax.plot(t, pred_c[col][row], color='tomato', linestyle='--', label='Pred')
            if col == 0:
                ax.set_ylabel(name, fontsize=8)
            if row == 0:
                ax.set_title(f'Sim {idx}', fontsize=9)
            if row == nrows - 1:
                ax.set_xlabel('Time step', fontsize=8)
            if col == ncols - 1 and row == 0:
                ax.legend(loc='upper right', fontsize=7)
    fig.suptitle(f'Autoenc-baseline — continuous waveforms{title_suffix}  (epoch {enc_ckpt["epoch"]})', fontsize=12)
    plt.tight_layout()
    return fig

def plot_valves(indices, pred_v, gt_v, title_suffix=''):
    ncols = len(indices)
    fig, axes = plt.subplots(4, ncols, figsize=(6 * ncols, 8), sharex=True)
    for col, idx in enumerate(indices):
        for row, name in enumerate(VALVE_NAMES):
            ax = axes[row, col]
            ax.step(t, gt_v[col][row], color='steelblue', where='post', label='GT')
            ax.step(t, (pred_v[col][row] > 0).astype(float), color='tomato', where='post', linestyle='--', label='Pred')
            ax.set_ylim(-0.1, 1.1)
            if col == 0:
                ax.set_ylabel(name, fontsize=8)
            if row == 0:
                ax.set_title(f'Sim {idx}', fontsize=9)
            if row == 3:
                ax.set_xlabel('Time step', fontsize=8)
            if col == ncols - 1 and row == 0:
                ax.legend(loc='upper right', fontsize=7)
    fig.suptitle(f'Autoenc-baseline — valve signals{title_suffix}  (epoch {enc_ckpt["epoch"]})', fontsize=12)
    plt.tight_layout()
    return fig

plot_continuous(SIM_INDICES, pred_conts, gt_conts)
plt.show()

In [ ]:
plot_valves(SIM_INDICES, pred_valves, gt_valves)
plt.show()

## Waveforms — 3 random test examples (saved)

In [ ]:
rng = np.random.default_rng()
rand_indices = sorted(rng.choice(N_SIMS, size=3, replace=False).tolist())
print(f'Random sim indices: {rand_indices}')

rand_pred_conts, rand_pred_valves, rand_gt_conts, rand_gt_valves = run_samples(rand_indices)

ids_str = '_'.join(str(i) for i in rand_indices)

fig_cont = plot_continuous(rand_indices, rand_pred_conts, rand_gt_conts, title_suffix=f' [sims {rand_indices}]')
fig_cont.savefig(FIGURES_DIR / f'cont_sims_{ids_str}.png', dpi=100, bbox_inches='tight')
plt.show()

fig_valve = plot_valves(rand_indices, rand_pred_valves, rand_gt_valves, title_suffix=f' [sims {rand_indices}]')
fig_valve.savefig(FIGURES_DIR / f'valve_sims_{ids_str}.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Saved to {FIGURES_DIR}/cont_sims_{ids_str}.png and valve_sims_{ids_str}.png')